In [1]:
!pip install fastapi uvicorn requests python-dotenv scikit-learn pandas joblib


In [2]:
HEALTH_CHAIN_API_KEY=""

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv("HEALTH_CHAIN_API_KEY")

In [4]:
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel
import requests
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import joblib


In [5]:
app = FastAPI(title="ML Health Prediction Engine")


In [6]:
data = pd.DataFrame({
    "sleep_hours": [4, 6, 8, 5, 7, 3, 9, 2],
    "water_intake": [0, 1, 2, 0, 2, 0, 2, 0],  # 0=low,1=medium,2=high
    "exercise": [0, 2, 5, 1, 4, 0, 6, 0],
    "symptoms": [1, 0, 0, 1, 0, 1, 0, 1],      # 1=yes, 0=no
    "risk": [2, 1, 0, 2, 1, 2, 0, 2]           # 0=Low,1=Moderate,2=High
})

X = data[["sleep_hours", "water_intake", "exercise", "symptoms"]]
y = data["risk"]


In [7]:
model = RandomForestClassifier()
model.fit(X, y)

# Save model (optional)
joblib.dump(model, "health_model.pkl")


['health_model.pkl']

In [8]:
class UserHealthData(BaseModel):
    sleep_hours: float
    water_intake: str
    exercise_per_week: int
    symptoms: str = ""

In [9]:
def preprocess(data: UserHealthData):

    water_map = {
        "low": 0,
        "medium": 1,
        "high": 2
    }

    return [[
        data.sleep_hours,
        water_map.get(data.water_intake.lower(), 0),
        data.exercise_per_week,
        1 if data.symptoms else 0
    ]]


In [10]:
def predict_risk(data: UserHealthData):

    processed = preprocess(data)
    pred = model.predict(processed)[0]

    labels = {
        0: "Low Risk",
        1: "Moderate Risk",
        2: "High Risk"
    }

    return labels[pred]


In [11]:
def explain_prediction(data, risk):

    prompt = f"""
    Explain this health risk level simply and safely.

    User:
    Sleep: {data.sleep_hours}
    Water: {data.water_intake}
    Exercise: {data.exercise_per_week}
    Symptoms: {data.symptoms}

    Risk: {risk}

    Give short explanation and 2 tips.
    """

    url = "https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent"

    headers = {"Content-Type": "application/json"}
    params = {"key": GEMINI_API_KEY}

    payload = {
        "contents": [
            {"role": "user", "parts": [{"text": prompt}]}
        ]
    }

    try:
        response = requests.post(url, headers=headers, params=params, json=payload)
        result = response.json()
        return result["candidates"][0]["content"]["parts"][0]["text"]
    except:
        return "Prediction generated (AI explanation unavailable)."


In [12]:
@app.post("/predict")
def predict(user_data: UserHealthData):

    risk = predict_risk(user_data)
    explanation = explain_prediction(user_data, risk)

    return JSONResponse(content={
        "status": "success",
        "risk_level": risk,
        "explanation": explanation
    })


In [14]:
@app.get("/")
def root():
    return {"message": "ML Health Prediction Engine running "}

In [15]:
class Dummy:
    sleep_hours = 4
    water_intake = "low"
    exercise_per_week = 0
    symptoms = "fatigue"

print(predict_risk(Dummy))
print(explain_prediction(Dummy, predict_risk(Dummy)))

High Risk
Prediction generated (AI explanation unavailable).


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
